# Che cosa si può calcolare

Il codice del capitolo [«Che cosa si può calcolare»](https://book.paithon.it/main/Introduzione/calcolabile.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Che cosa si può calcolare

[Leggi la pagina](https://book.paithon.it/main/Introduzione/calcolabile.html)


### Una macchina con un nastro e una matita


In [ ]:
def esegui(regole, nastro, stato, passi_max=1000, mostra=False):
    """Una macchina di Turing. `regole` dice, per ogni coppia (stato, simbolo
    letto), che cosa scrivere, dove spostare la testina (-1 a sinistra, +1 a
    destra, 0 ferma) e in quale stato passare. Senza una regola la macchina si
    ferma. Restituisce il nastro, i passi fatti e se si è fermata."""
    caselle = dict(enumerate(nastro))            # le caselle scritte; il resto è bianco
    testina = 0
    leggi = lambda: "".join(caselle.get(i, " ")
                            for i in range(min(caselle, default=0),
                                           max(caselle, default=-1) + 1)).strip()
    for passo in range(passi_max):
        letto = caselle.get(testina, " ")
        if (stato, letto) not in regole:
            return leggi(), passo, True
        caselle[testina], sposta, stato = regole[(stato, letto)]
        testina += sposta
        if mostra:
            print(f"  dopo il passo {passo + 1}: {leggi():<5} stato {stato}")
    return leggi(), passi_max, False

# aggiungere uno a un numero scritto in binario: si va in fondo, poi si torna
# indietro cambiando gli 1 in 0 finché non si trova uno 0 o una casella bianca
incremento = {
    ("vai", "0"): ("0", +1, "vai"),
    ("vai", "1"): ("1", +1, "vai"),
    ("vai", " "): (" ", -1, "riporto"),
    ("riporto", "1"): ("0", -1, "riporto"),
    ("riporto", "0"): ("1", 0, "fine"),
    ("riporto", " "): ("1", 0, "fine"),
}
nastro, passi, fermata = esegui(incremento, "1011", "vai", mostra=True)
print(f"1011 + 1 = {nastro}, in {passi} passi")
print(f"111 + 1 = {esegui(incremento, '111', 'vai')[0]}")

# la prima macchina dell'articolo di Turing (1936): quattro stati, b, c, e, k,
# che stampano 0 e 1 alternati lasciando una casella vuota fra una cifra e l'altra
macchina_I = {
    ("b", " "): ("0", +1, "c"),
    ("c", " "): (" ", +1, "e"),
    ("e", " "): ("1", +1, "k"),
    ("k", " "): (" ", +1, "b"),
}
nastro, passi, fermata = esegui(macchina_I, "", "b", passi_max=16)
print(f"macchina I dopo {passi} passi: {nastro!r}; si è fermata? {fermata}")

### La domanda che nessun programma risolve


In [ ]:
def collatz(n):
    """Pari: dimezza. Dispari: triplica e aggiungi uno. Fino a 1, se ci arriva."""
    passi, massimo = 0, n
    while n != 1:
        n = n // 2 if n % 2 == 0 else 3 * n + 1
        passi, massimo = passi + 1, max(massimo, n)
    return passi, massimo

for n in (6, 7, 27):
    passi, massimo = collatz(n)
    print(f"partendo da {n}: arriva a 1 in {passi} passi, salendo fino a {massimo}")
record = max(range(1, 10_000), key=lambda n: collatz(n)[0])
print(f"sotto 10 000 il cammino più lungo parte da {record}: {collatz(record)[0]} passi")

### Una frase che parla di sé


In [ ]:
# il gioco MIU di Hofstadter: si parte da MI e si applicano quattro regole
def figli(s):
    if s.endswith("I"):
        yield s + "U"                            # regola I:   xI  -> xIU
    yield s + s[1:]                              # regola II:  Mx  -> Mxx
    for i in range(len(s) - 2):
        if s[i:i + 3] == "III":
            yield s[:i] + "U" + s[i + 3:]        # regola III: III -> U
    for i in range(len(s) - 1):
        if s[i:i + 2] == "UU":
            yield s[:i] + s[i + 2:]              # regola IV:  UU  -> niente

teoremi, frontiera = {"MI"}, ["MI"]
while frontiera:
    nuovi = [t for s in frontiera for t in figli(s) if len(t) <= 12 and t not in teoremi]
    teoremi.update(nuovi)
    frontiera = list(dict.fromkeys(nuovi))
print(f"stringhe ottenute da MI, lunghe al più 12: {len(teoremi)}")
print(f"MU fra queste? {'MU' in teoremi}")
resti = {t.count('I') % 3 for t in teoremi}
print(f"resti del numero di I diviso 3, su tutte: {sorted(resti)}")